# Cell 1: Environment Setup

In [ ]:
import os
import math
import copy
import time
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
)


# ==============================================================================
# v5 CONSOLIDATED PIPELINE — every choice below is FIXED (not switchable).
# The audio feature path in Cell 4 hardcodes exactly these settings; SEED is
# the single source for all seeding in this notebook.
# ==============================================================================
SEED = 42
PIPELINE_CONFIG = {
    "Audio bin selection": "strongest64 (top-64 mean |rFFT| bins, TRAINING audio only, frozen for val/test)",
    "Audio windowing":     "none (raw 1536-sample chunk -> rfft; best tested option, see Cell 4b diagnostic)",
    "Audio compression":   "sqrt(magnitude)",
    "Audio normalization": "none",
    "Classification loss": "class-weighted cross-entropy (inverse TRAIN-set class frequency)",
    "Seed":                f"{SEED} (random / numpy / torch / cuda, cuDNN deterministic)",
}
print("=" * 90)
print("  AEROSYNC-MAMBA v5 — CONSOLIDATED PIPELINE CONFIGURATION")
print("=" * 90)
for _k, _v in PIPELINE_CONFIG.items():
    print(f"  • {_k:<20}: {_v}")
print("=" * 90)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ==============================================================================
# FIX 2: install the REAL mamba_ssm package (selective-scan SSM kernels).
#
# STRATEGY (in order):
#   1. PREBUILT WHEEL from the causal-conv1d / mamba-ssm GitHub releases, matched
#      to this exact Python / torch / CUDA / C++ ABI build. This is the ONLY
#      reliable path on Kaggle — building from source routinely hangs or OOMs
#      the kernel because Kaggle's toolchain doesn't line up with what these
#      packages' setup.py expects.
#   2. Capped from-source build (240s timeout, output captured) as a fallback
#      if no matching prebuilt wheel exists for this environment.
#   3. Pure-PyTorch fallback block (Cell 2) if both of the above fail — this is
#      the safety net that guarantees the notebook can NEVER hang/crash here,
#      but every mode in the frequency-bin sweep shares whichever backbone this
#      cell resolves to, so the 6-way comparison stays apples-to-apples either way.
# ==============================================================================
MAMBA_INSTALL_TIMEOUT_S = 240

def _detect_env_tags():
    """Python tag, torch major.minor, CUDA major.minor, C++11 ABI flag — the
    axes prebuilt causal-conv1d / mamba-ssm wheels are keyed on."""
    import sys
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_ver = torch.version.cuda  # e.g. "12.1", or None on CPU-only torch
    try:
        abi_flag = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except Exception:
        abi_flag = bool(getattr(torch, "compiled_with_cxx11_abi", lambda: False)())
    return py_tag, torch_mm, cuda_ver, abi_flag

def _github_releases(repo, timeout_s=20):
    import urllib.request, json as _json
    url = f"https://api.github.com/repos/{repo}/releases"
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        return _json.load(resp)

def _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag):
    """Best-matching .whl asset URL from `repo`'s GitHub releases, or None."""
    try:
        releases = _github_releases(repo)
    except Exception as e:
        print(f"    could not query github.com/{repo}/releases: {e!r}")
        return None

    abi_str = f"cxx11abi{'TRUE' if abi_flag else 'FALSE'}"
    cuda_major = cuda_ver.split(".")[0] if cuda_ver else None
    cuda_tag_exact = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else None   # e.g. "cu121"
    exact, major_only = [], []

    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not name.endswith(".whl"):
                continue
            if py_tag not in name or abi_str not in name or f"torch{torch_mm}" not in name:
                continue
            if cuda_tag_exact and cuda_tag_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major and f"cu{cuda_major}" in name:
                major_only.append(asset["browser_download_url"])

    chosen = (exact or major_only)
    return chosen[0] if chosen else None

def _pip_run(args, timeout_s=MAMBA_INSTALL_TIMEOUT_S):
    """pip invocation that can NEVER hang or crash the kernel: hard timeout,
    output captured (not streamed — a runaway nvcc/ninja build can print
    megabytes), every exception caught."""
    import subprocess, sys
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + args
    try:
        proc = subprocess.run(cmd, timeout=timeout_s, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            tail = (proc.stderr or proc.stdout or "")[-500:]
            print(f"    ⚠️  pip install {args} failed (exit {proc.returncode}); tail:\n{tail}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"    ⚠️  pip install {args} exceeded {timeout_s}s — killed to avoid hanging/OOM-ing "
              f"the kernel; treating as failed.")
        return False
    except Exception as e:
        print(f"    ⚠️  pip install {args} raised {e!r}; treating as failed.")
        return False

def _install_via_prebuilt_wheel(pip_name, repo, py_tag, torch_mm, cuda_ver, abi_flag):
    url = _find_prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, abi_flag)
    if url is None:
        print(f"    no matching prebuilt wheel found for {pip_name} "
              f"(py={py_tag}, torch={torch_mm}, cuda={cuda_ver}, cxx11abi={abi_flag})")
        return False
    print(f"    found prebuilt wheel: {url}")
    return _pip_run([url, "--no-deps"])

_MAMBA_INSTALL_METHOD = "not attempted (no CUDA GPU)"

if torch.cuda.is_available():
    try:
        import mamba_ssm  # noqa: F401
        print("mamba_ssm already installed.")
        _MAMBA_INSTALL_METHOD = "already installed"
    except ImportError:
        py_tag, torch_mm, cuda_ver, abi_flag = _detect_env_tags()
        print(f"Environment: python={py_tag} | torch={torch_mm} | cuda={cuda_ver} | cxx11abi={abi_flag}")
        print("Installing real mamba_ssm + causal-conv1d (selective-scan SSM kernels)...")

        print("  [1/3] trying prebuilt wheels (no compilation — the reliable path on Kaggle)...")
        ok1 = _install_via_prebuilt_wheel("causal-conv1d", "Dao-AILab/causal-conv1d", py_tag, torch_mm, cuda_ver, abi_flag)
        ok2 = _install_via_prebuilt_wheel("mamba-ssm", "state-spaces/mamba", py_tag, torch_mm, cuda_ver, abi_flag) if ok1 else False

        if ok1 and ok2:
            _MAMBA_INSTALL_METHOD = "prebuilt wheel"
        else:
            print(f"  [2/3] no complete prebuilt-wheel match — falling back to a capped "
                  f"from-source build ({MAMBA_INSTALL_TIMEOUT_S}s timeout per package)...")
            # --no-build-isolation avoids re-resolving/reinstalling torch inside an isolated
            # build env, a common cause of causal-conv1d's build spinning forever on Kaggle.
            ok1b = _pip_run(["causal-conv1d>=1.2.0", "--no-build-isolation"])
            ok2b = _pip_run(["mamba-ssm", "--no-build-isolation"]) if ok1b else False
            _MAMBA_INSTALL_METHOD = "source build" if (ok1b and ok2b) else "failed"

        print(f"  [3/3] resolution: {_MAMBA_INSTALL_METHOD}")
        if _MAMBA_INSTALL_METHOD in ("prebuilt wheel", "source build"):
            print(f"✅ causal-conv1d + mamba-ssm installed via {_MAMBA_INSTALL_METHOD}.")
        else:
            print("⚠️  mamba_ssm install did not complete — the pure-PyTorch FALLBACK block will be "
                  "used (see Cell 2 / Cell 6 output). This does not affect the frequency-bin sweep: "
                  "every mode uses the same backbone, so the 6-way comparison is still apples-to-apples.")
else:
    print("⚠️  No CUDA GPU detected — mamba_ssm requires CUDA, so the pure-PyTorch FALLBACK block will be used (see Cell 2 / Cell 6 output).")


# ==============================================================================
# GLOBAL SEEDING — python `random`, numpy, torch CPU + every CUDA device, plus
# deterministic cuDNN. The mission split in Cell 4 keeps its own
# random.seed(42); this covers everything else (weight init, dropout, DataLoader
# shuffle order). set_all_seeds() is called again right before model
# construction in Cell 6 and before the unweighted comparison run (Cell 10).
# Note: mamba_ssm's CUDA selective-scan backward uses atomic adds, so GPU runs
# can still differ in the last few decimal places even with identical seeds.
# ==============================================================================
def set_all_seeds(seed: int = SEED) -> int:
    """Reset python / numpy / torch (CPU + all CUDA devices) RNGs to `seed`."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return seed

set_all_seeds(SEED)
print(f"✅ Seeds set -> random={SEED}, numpy={SEED}, torch={SEED}, cuda={SEED} (cuDNN deterministic)")

# Cell 2: Bi-Modal AeroSync-Mamba Architecture (HTT + LACA + Mamba)

In [ ]:
# --- Fusion backbone resolution: REAL mamba_ssm (CUDA selective-scan) vs. pure-PyTorch FALLBACK ---
_MAMBA_CLASS = None
_FUSION_BACKBONE_REASON = None
try:
    if torch.cuda.is_available():
        from mamba_ssm import Mamba as _MAMBA_CLASS
        _FUSION_BACKBONE_REASON = "mamba_ssm.Mamba (REAL selective-scan SSM: hidden state h_t, learned A/B/C, discretized Δ) — CUDA GPU detected"
    else:
        _FUSION_BACKBONE_REASON = "FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — no CUDA GPU detected; mamba_ssm requires CUDA"
except ImportError as _mamba_import_err:
    _FUSION_BACKBONE_REASON = f"FALLBACK (VectorizedSSMBlock, pure PyTorch, NO SSM recurrence) — mamba_ssm import failed: {_mamba_import_err}"

print(f"\n{'='*70}\nFUSION BACKBONE RESOLUTION: {_FUSION_BACKBONE_REASON}\n{'='*70}\n")


# --- 1. Continuous Temporal Positional Embedding & HTT ---
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = 64):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        fourier = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return self.mlp(fourier)


class HeterogeneousTemporalTokenization(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        # 0: Audio, 1: Sensor Telemetry
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bar: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z_bar.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        phi_tau = self.time_embedding(tau)
        return self.norm(z_bar + e_m + phi_tau)


# --- 2. Bi-Modal LACA (Audio <-> Telemetry Alignment) ---
class PairwiseLACA(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        B_mn = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn_weights = F.softmax(scores + self.beta * B_mn, dim=-1)
        out = torch.matmul(attn_weights, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)
        H_tilde_mn = self.out_proj(out)

        A_mn = attn_weights.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (A_mn * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return H_tilde_mn, A_mn, expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4, beta: float = 1.0):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, num_heads, beta)
        self.laca_sa = PairwiseLACA(d_model, num_heads, beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, _, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, _, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)

        H_tilde_a = Ha + self.gamma_as * H_a_from_s
        H_tilde_s = Hs + self.gamma_sa * H_s_from_a
        return H_tilde_a, H_tilde_s, {"lag_as": lag_as, "lag_sa": lag_sa}


# --- 3a. FALLBACK: pure-PyTorch gated-conv block (NOT a real SSM) ---
# Used ONLY when mamba_ssm / CUDA is unavailable (see resolution printed
# above). This block has NO hidden state h_t, NO A/B/C matrices, and NO
# discretization step — it is a depthwise-conv + gated-MLP block, kept only
# as a last-resort fallback so the notebook still runs on CPU-only machines.
class VectorizedSSMBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_inner = int(expand * d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.mixer = nn.Sequential(
            nn.Linear(self.d_inner, self.d_inner),
            nn.GELU(),
            nn.Linear(self.d_inner, self.d_inner)
        )
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        residual = x
        x_norm = self.norm(x)
        xz = self.in_proj(x_norm)
        x_branch, z_branch = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)

        y = self.mixer(x_conv) * F.silu(z_branch)
        return residual + self.out_proj(y)


# --- 3b. REAL Mamba block (genuine selective-scan SSM) ---
# Thin wrapper around mamba_ssm.Mamba, which implements the actual S6
# recurrence (hidden state h_t, learned continuous A/B/C matrices, and a
# discretization step Δ), wrapped in the same pre-norm + residual pattern as
# the fallback block above so it is a drop-in replacement.
class RealMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_conv: int = 4, expand: int = 2, d_state: int = 16):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = _MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


# --- 4. Full Bi-Modal AeroSync-Mamba Model ---
class BiModalAeroSyncMamba(nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        n_classes: int = 5,
        num_mamba_blocks: int = 4,
        dropout: float = 0.15,
        temporal_bias_scale: float = 1.0,
        audio_in_dim: int = 64,
        sensor_in_dim: int = 6,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        # Sensor 1D-CNN encoder
        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(sensor_in_dim, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        # Audio feature projector
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )

        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model=d_model, num_heads=4, beta=temporal_bias_scale)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # --- FIX 2: use the REAL mamba_ssm block when available, otherwise the
        # pure-PyTorch fallback (see resolution printed at the top of this cell,
        # and the unmissable print in Cell 6 right after model construction). ---
        if _MAMBA_CLASS is not None:
            self.mamba_blocks = nn.ModuleList([
                RealMambaBlock(d_model=d_model, d_conv=4, expand=2) for _ in range(num_mamba_blocks)
            ])
        else:
            self.mamba_blocks = nn.ModuleList([
                VectorizedSSMBlock(d_model=d_model, d_conv=4, expand=2) for _ in range(num_mamba_blocks)
            ])
        self.fusion_backbone_reason = _FUSION_BACKBONE_REASON

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )

        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]

        # 1. Modality Encoders
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        # 2. HTT (0: Audio, 1: Sensor)
        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)

        # 3. Bi-Modal LACA Alignment
        H_tilde_a, H_tilde_s, aux = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        # 4. Cross-Modal Mamba Fusion: [ H_tilde_a ; [SEP] ; H_tilde_s ]
        sep = self.sep_token.expand(B, -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)

        F_seq = H_fused
        for blk in self.mamba_blocks:
            F_seq = blk(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled_reps = {"z_a": za_p, "z_s": zs_p}
        aux_preds = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled_reps, aux_preds, aux

print("Bi-Modal AeroSync-Mamba model class defined successfully!")

# Cell 3: Bi-Modal Loss Function (Classification + Audio-IMU InfoNCE + KL)

In [ ]:
class BiModalAeroSyncLoss(nn.Module):
    def __init__(self, lambda_align=0.1, lambda_cons=0.1, temperature=0.07, class_weights=None):
        super().__init__()
        self.lambda_align = lambda_align
        self.lambda_cons = lambda_cons
        self.temperature = temperature
        # --- FIX 4: class-weighted classification loss. `class_weights` is an
        # inverse-frequency tensor computed from REAL training-set class counts
        # in Cell 6 (printed there for verification), not hardcoded here. ---
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        z1_norm = F.normalize(z1, p=2, dim=-1)
        z2_norm = F.normalize(z2, p=2, dim=-1)
        sim = torch.matmul(z1_norm, z2_norm.t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_preds):
        # 1. Fault Classification Loss
        loss_cls = self.ce(logits, targets)

        # 2. Audio-Sensor Contrastive Alignment Loss
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])

        # 3. Auxiliary Consistency Loss
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_preds["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_preds["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0

        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": float(loss_cls.item()), "loss_align": float(loss_align.item())}

def compute_all_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    return {"accuracy": acc, "macro_f1": f1, "balanced_accuracy": bal_acc, "macro_auroc": auroc}

print("Loss & Metrics helper ready!")

# Cell 4 (Self-Downloading & Preprocessing): Real TII UAV Dataset


In [ ]:
import os
import json
import random
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import torch

DATASET_BASE = Path("/kaggle/working/tii_uav_dataset")
DATASET_ROOT = DATASET_BASE / "Dataset"
CACHE_DIR = Path("/kaggle/working/preprocessed_bimodal_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically clone the official repository if not present
if not DATASET_ROOT.exists():
    print("📥 Cloning official TII UAV Realistic Fault Dataset from GitHub...")
    !git clone --depth 1 https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git /kaggle/working/tii_uav_dataset
    print("✅ Dataset cloned successfully!")

_chunk_duration_log = []
_fft_shape_printed = False  # verification: print the FFT feature shape once

N_AUDIO_BINS = 64  # frozen output width of the audio feature vector
SELECTED_BIN_INDICES = None  # frozen strongest-64 bin indices (set below, train-only)

# 2. Real Flight Mission Parser (6-Axis IMU + Audio)
def parse_real_mission(mission_dir: Path):
    sensor_matches = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_matches  = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_matches or not audio_matches:
        return None

    # 2.1 Parse Real IMU
    sensor_times, accel_data, gyro_data = [], [], []
    with open(sensor_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel_data.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro_data.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue

    if len(sensor_times) < 50: return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel_data, dtype=np.float32), np.array(gyro_data, dtype=np.float32)], axis=-1)

    # 2.2 Parse Real Audio
    audio_chunks = []
    with open(audio_matches[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                samples = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(samples) > 0: audio_chunks.append(samples)
            except Exception:
                continue

    if len(audio_chunks) < 10: return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    # --- FIX: derive REAL per-chunk audio timestamps ---
    # `recording_started_time` is a constant repeated on every line (not a
    # real per-chunk timestamp), and the previous fixed 0.10s/chunk guess
    # doesn't match the data (empirically ~17-19ms/chunk). Instead, derive
    # chunk duration from this mission's REAL, IMU-verified flight duration,
    # since audio and IMU are recorded within the same continuous rosbag
    # session.
    mission_duration_s = sensor_times[-1] - sensor_times[0]  # already zero-based
    n_chunks = len(audio_chunks)
    chunk_duration_s = mission_duration_s / n_chunks
    audio_times = np.arange(n_chunks, dtype=np.float64) * chunk_duration_s

    global _chunk_duration_log
    _chunk_duration_log.append(chunk_duration_s)

    return {
        "sensor_times": sensor_times,
        "telemetry": telemetry,
        "audio_times": audio_times,
        "audio_chunks": audio_chunks,
    }


# 2.3 Strongest-64 FFT bin selection (TRAINING-SET audio only, unsupervised, leakage-safe)
def select_strongest_audio_bins(train_mission_dirs, n_bins=N_AUDIO_BINS):
    """
    Selects the `n_bins` FFT magnitude bins with the highest mean magnitude,
    computed ONLY from training-set audio (raw waveform energy — never class
    labels). Called once, before any train/val/test window slicing, so the
    exact same frozen indices are reused identically across all three splits.

    A controlled 6-way comparison (first64 / middle64 / high64 / logspaced64 /
    compressed_full / strongest64) found this strategy beats the previous
    first-64-bins baseline on accuracy, macro-F1, balanced accuracy, and
    macro-AUROC by ~+2.4 to +2.8 points, so this is now the single permanent
    baseline — not a switchable mode.
    """
    print("[STRONGEST-64] Computing bin-selection statistics from TRAINING-SET audio only "
          f"({len(train_mission_dirs)} training missions) — BEFORE any train/validation/test "
          "window preprocessing.")

    bin_magnitude_sum = None
    bin_count = 0
    fft_len_ref = None
    local_chunk_durations = []

    for m_dir in tqdm(train_mission_dirs, desc="Bin selection (train-only audio scan)", leave=False):
        mission_data = parse_real_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        a_times = mission_data["audio_times"]
        if len(a_times) > 1:
            local_chunk_durations.append((a_times[-1] - a_times[0]) / (len(a_times) - 1))

        mag = np.abs(np.fft.rfft(audio, axis=1))  # (n_chunks, fft_len), unsupervised energy
        if fft_len_ref is None:
            fft_len_ref = mag.shape[1]
        min_len = min(fft_len_ref, mag.shape[1])
        mag = mag[:, :min_len]
        if bin_magnitude_sum is not None and len(bin_magnitude_sum) > min_len:
            bin_magnitude_sum = bin_magnitude_sum[:min_len]
        fft_len_ref = min_len

        chunk_sum = mag.sum(axis=0)
        bin_magnitude_sum = chunk_sum if bin_magnitude_sum is None else bin_magnitude_sum + chunk_sum
        bin_count += mag.shape[0]

    assert bin_magnitude_sum is not None and bin_count > 0, (
        "STRONGEST-64 FAILED: no training-set audio chunks were available for bin selection."
    )

    mean_magnitude = bin_magnitude_sum / bin_count  # unsupervised magnitude/energy criterion
    top_bins = np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)

    assert len(top_bins) == n_bins, (
        f"STRONGEST-64 FAILED: selected {len(top_bins)} bins, expected {n_bins}"
    )

    # --- Hz labeling ---
    # This notebook does not document a raw microphone sample rate anywhere,
    # so we estimate an effective sample rate from quantities the notebook
    # already derives from real data: samples-per-audio-buffer (inverted from
    # the rFFT length) and the real per-mission inter-buffer timing computed
    # above (same derivation `parse_real_mission` already uses for
    # `chunk_duration_s`). This is a data-derived estimate used only for
    # human-readable labeling below — it never affects which bins are
    # selected or how features are built.
    samples_per_buffer = fft_len_ref * 2 - 2
    est_sample_rate_hz = (
        samples_per_buffer / np.mean(local_chunk_durations) if local_chunk_durations else None
    )

    print(f"[STRONGEST-64] Selected {n_bins} bins from {bin_count} training-set audio chunks "
          f"across {len(train_mission_dirs)} missions (FFT length = {fft_len_ref}).")
    print(f"[STRONGEST-64] Selected bin indices (ascending): {top_bins.tolist()}")
    if est_sample_rate_hz:
        bin_width_hz = est_sample_rate_hz / samples_per_buffer
        print(f"[STRONGEST-64] Estimated audio sample rate ≈ {est_sample_rate_hz:.1f} Hz "
              "(derived from audio buffer length and real inter-buffer timing across training "
              "missions; the dataset does not document a mic sample rate directly, so this is "
              "a data-derived estimate for labeling only, not a hardcoded spec).")
        print("[STRONGEST-64] Selected bin index -> approx. frequency (Hz):")
        for idx in top_bins.tolist():
            print(f"    bin {idx:3d}  ->  {idx * bin_width_hz:8.1f} Hz")
    else:
        print("[STRONGEST-64] Could not estimate a sample rate (insufficient timing data); "
              "printing bin indices only.")

    return top_bins


# 2.4 Audio feature extraction — the single, permanent path (v5):
#     raw chunk -> rfft (no window) -> |.| -> frozen strongest-64 bins -> sqrt
#     (no normalization). Consolidated from the v4 sequential experiment:
#     windowing "none" (best tested), compression "sqrt" (Stage B winner),
#     normalization "none" (Stage C winner).
def compute_audio_features(audio_chunks):
    """
    audio_chunks : (n_chunks, chunk_len) raw waveform chunks of one mission
    returns      : (n_chunks, 64) float32 = sqrt(|rfft|) at the frozen
                   SELECTED_BIN_INDICES (computed once from TRAINING audio by
                   select_strongest_audio_bins; never recomputed here).
    """
    assert SELECTED_BIN_INDICES is not None, (
        "FEATURES FAILED: strongest-64 bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any feature extraction."
    )

    audio_fft_mag = np.abs(np.fft.rfft(audio_chunks, axis=1))  # (n_chunks, chunk_len//2 + 1), REAL FFT
    n_chunks_mission = audio_fft_mag.shape[0]
    n_bins_available = audio_fft_mag.shape[1]
    valid_idx = SELECTED_BIN_INDICES[SELECTED_BIN_INDICES < n_bins_available]

    a_feats = np.zeros((n_chunks_mission, N_AUDIO_BINS), dtype=np.float32)
    a_feats[:, :len(valid_idx)] = np.sqrt(audio_fft_mag[:, valid_idx]).astype(np.float32)

    assert a_feats.shape == (n_chunks_mission, N_AUDIO_BINS), (
        f"FEATURES FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {N_AUDIO_BINS})"
    )
    return a_feats


# 3. Slicing into 2.56s Diagnostic Samples
def slice_real_bimodal(mission_data, class_label, window_sec=2.56, step_sec=1.28, Ta=64, Ts=128):
    global _fft_shape_printed
    s_times = mission_data["sensor_times"]
    telemetry = mission_data["telemetry"]
    a_times = mission_data["audio_times"]
    audio = mission_data["audio_chunks"]

    assert SELECTED_BIN_INDICES is not None, (
        "SLICE FAILED: strongest-64 bin indices must be computed from training audio "
        "(select_strongest_audio_bins) before any window preprocessing."
    )

    # --- strongest64 audio features: FFT magnitude spectrum, frozen top-64 bins ---
    # Previously: the first 64 FFT bins (low frequencies only) were used
    # unconditionally for every split. Now: the 64 bins with the highest mean
    # magnitude in TRAINING-set audio (computed once in
    # select_strongest_audio_bins, unsupervised, leakage-safe) are frozen and
    # reused identically for train, validation, and test.
    # FFT (no window) -> frozen strongest-64 bins -> sqrt, via compute_audio_features (2.4).
    a_feats = compute_audio_features(audio)
    n_chunks_mission = audio.shape[0]

    assert a_feats.shape == (n_chunks_mission, N_AUDIO_BINS), (
        f"STRONGEST-64 FAILED: audio feature array shape is {a_feats.shape}, "
        f"expected (n_chunks={n_chunks_mission}, {N_AUDIO_BINS})"
    )

    if not _fft_shape_printed:
        print(f"[STRONGEST-64 VERIFICATION] audio chunk shape (raw waveform): {audio.shape} "
              f"| np.fft.rfft output shape: {(audio.shape[0], audio.shape[1] // 2 + 1)} "
              f"| final a_feats (sqrt-compressed, frozen strongest-64 bins) shape: {a_feats.shape}")
        print("[STRONGEST-64 VERIFICATION] Confirmed: features were produced via np.fft.rfft "
              "(frequency-domain magnitude spectrum) using the frozen strongest-64 bin indices, "
              "NOT the first 64 bins.")
        _fft_shape_printed = True

    max_time = min(s_times[-1], a_times[-1])
    samples = []
    t_start = 0.0

    while t_start + window_sec <= max_time:
        t_end = t_start + window_sec
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < 10:
            t_start += step_sec
            continue

        # --- FIX 1: REAL per-window sensor timestamps ---
        # Previously: target_s_t = np.linspace(t_start, t_end, Ts) — a synthetic
        # uniform grid that is IDENTICAL for every window (window_sec is fixed),
        # so nothing real ever reached the model downstream. Now: pick Ts
        # evenly-spaced INDICES into the actual raw IMU sample timestamps that
        # fall inside this window. Real IMU logging is not perfectly uniform, so
        # these timestamps genuinely vary window-to-window and mission-to-mission.
        s_idx_in_window = np.where(s_mask)[0]
        real_s_pos = np.linspace(0, len(s_idx_in_window) - 1, Ts).round().astype(int)
        real_s_idx = s_idx_in_window[real_s_pos]
        target_s_t = s_times[real_s_idx]  # REAL raw sensor timestamps (seconds, mission-relative)

        s_window = np.zeros((Ts, 6), dtype=np.float32)
        for c in range(6):
            s_window[:, c] = np.interp(target_s_t, s_times[s_mask], telemetry[s_mask, c])

        # --- FIX 1: REAL per-window audio timestamps ---
        # audio_times itself is a per-mission uniform grid derived from the
        # mission's real chunk_duration_s (see parse_real_mission above), which
        # genuinely varies mission-to-mission (see the chunk-duration min/max
        # printed at the end of this cell). We pick real indices from it within
        # this window rather than an independent synthetic linspace.
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx_in_window = np.where(a_mask)[0]
            real_a_pos = np.linspace(0, len(a_idx_in_window) - 1, Ta).round().astype(int)
            real_a_idx = a_idx_in_window[real_a_pos]
            target_a_t = a_times[real_a_idx]  # REAL per-mission audio timestamps
        else:
            # Fallback only for the rare window with too few real audio samples
            target_a_t = np.linspace(t_start, t_end, Ta)

        a_window = np.zeros((Ta, N_AUDIO_BINS), dtype=np.float32)
        for c in range(N_AUDIO_BINS):
            a_window[:, c] = np.interp(target_a_t, a_times, a_feats[:, c], left=0.0, right=0.0)

        assert a_window.shape == (Ta, N_AUDIO_BINS), (
            f"STRONGEST-64 FAILED: window audio feature shape is {a_window.shape}, "
            f"expected ({Ta}, {N_AUDIO_BINS})"
        )

        samples.append({
            "x_a": a_window,
            "x_s": s_window,
            "label": int(class_label),
            "tau_a": (target_a_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "tau_s": (target_s_t - t_start).astype(np.float32),  # REAL, zero-based, seconds
            "window_sec": np.float32(window_sec),
        })
        t_start += step_sec

    return samples

# --- VERIFICATION: no leftover mode-switching in the audio feature path ---
import inspect
_feat_params = list(inspect.signature(compute_audio_features).parameters)
_slice_params = list(inspect.signature(slice_real_bimodal).parameters)
_switch_names = ("window_fn", "compression", "norm_stats", "normalization", "audio_mode", "mode")
_switch_params = [p for p in _feat_params + _slice_params if p in _switch_names]
try:
    _feat_src = inspect.getsource(compute_audio_features)
    _dead_branches = [t for t in ("hanning", "log1p", "log10", "zscore", "window_fn", "compression ==",
                                  "norm_stats", "elif") if t in _feat_src]
except (OSError, TypeError):
    _feat_src, _dead_branches = None, []
_stale_globals = [n for n in ("AUDIO_WINDOW_FNS", "AUDIO_COMPRESSIONS", "LOG_DB_EPS", "get_audio_window",
                              "compute_train_norm_stats") if n in globals()]

assert _feat_params == ["audio_chunks"], f"compute_audio_features still takes switches: {_feat_params}"
assert not _switch_params, f"mode-switching parameters remain: {_switch_params}"
assert not _dead_branches, f"dead condition branches remain in compute_audio_features: {_dead_branches}"
print(f"[NO-MODE-SWITCH CHECK] compute_audio_features{tuple(_feat_params)} | slice_real_bimodal has no "
      f"window/compression/normalization parameters | source scan for hann/log1p/log_db/zscore/elif "
      f"branches: {'clean' if _feat_src is not None else 'source unavailable (signature check only)'} ✅")
if _stale_globals:
    print(f"⚠️  stale globals from an older notebook are still in this kernel: {_stale_globals} "
          f"(unused by v5 — restart the kernel for a clean run)")
else:
    print("[NO-MODE-SWITCH CHECK] no experiment globals (window / compression / normalization options) defined ✅")

# 4. Mission-Level Stratification across all 99 Flights
class_missions = defaultdict(list)
for c_dir in sorted(DATASET_ROOT.iterdir()):
    if not c_dir.is_dir() or c_dir.name.startswith("."): continue
    try: c_label = int(c_dir.name)
    except ValueError: continue
    for m_dir in sorted(c_dir.iterdir()):
        if m_dir.is_dir() and not m_dir.name.startswith("."):
            class_missions[c_label].append(m_dir)

random.seed(42)

train_mission_map, val_mission_map, test_mission_map = {}, {}, {}
for c_label in sorted(class_missions.keys()):
    missions = class_missions[c_label]
    random.shuffle(missions)
    n_train = max(1, int(len(missions) * 0.70))
    n_val = max(1, int(len(missions) * 0.15))

    train_mission_map[c_label] = missions[:n_train]
    val_mission_map[c_label]   = missions[n_train:n_train + n_val]
    test_mission_map[c_label]  = missions[n_train + n_val:]

    print(f"Class {c_label} ({len(missions)} flights) -> "
          f"Train: {len(train_mission_map[c_label])}, "
          f"Val: {len(val_mission_map[c_label])}, "
          f"Test: {len(test_mission_map[c_label])}")

# --- Freeze strongest-64 bin indices from TRAINING missions only, BEFORE any
# train/val/test window preprocessing happens. ---
all_train_missions = [m for missions in train_mission_map.values() for m in missions]
SELECTED_BIN_INDICES = select_strongest_audio_bins(all_train_missions, n_bins=N_AUDIO_BINS)
_bin_fingerprint_at_selection = (
    SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist()
)

train_samples, val_samples, test_samples = [], [], []

print("\n--- Preprocessing TRAIN split (frozen strongest-64 bins) ---")
for c_label in sorted(class_missions.keys()):
    for m in tqdm(train_mission_map[c_label], desc=f"Train Class {c_label}", leave=False):
        p = parse_real_mission(m)
        if p: train_samples.extend(slice_real_bimodal(p, c_label))
_fp_train = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-64] Frozen bin indices applied @ TRAIN (first5, last5): {_fp_train} "
      f"| matches selection: {_fp_train == _bin_fingerprint_at_selection}")

print("\n--- Preprocessing VAL split (same frozen bin indices — NOT recomputed) ---")
for c_label in sorted(class_missions.keys()):
    for m in tqdm(val_mission_map[c_label], desc=f"Val Class {c_label}", leave=False):
        p = parse_real_mission(m)
        if p: val_samples.extend(slice_real_bimodal(p, c_label))
_fp_val = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-64] Frozen bin indices applied @ VAL   (first5, last5): {_fp_val} "
      f"| matches TRAIN: {_fp_val == _fp_train}")

print("\n--- Preprocessing TEST split (same frozen bin indices — NOT recomputed) ---")
for c_label in sorted(class_missions.keys()):
    for m in tqdm(test_mission_map[c_label], desc=f"Test Class {c_label}", leave=False):
        p = parse_real_mission(m)
        if p: test_samples.extend(slice_real_bimodal(p, c_label))
_fp_test = (SELECTED_BIN_INDICES[:5].tolist(), SELECTED_BIN_INDICES[-5:].tolist())
print(f"[STRONGEST-64] Frozen bin indices applied @ TEST  (first5, last5): {_fp_test} "
      f"| matches TRAIN: {_fp_test == _fp_train}")

# --- Final shape verification across all three splits ---
for split_name, split_samples in [("train", train_samples), ("val", val_samples), ("test", test_samples)]:
    if split_samples:
        x_a_shapes = {tuple(s["x_a"].shape) for s in split_samples}
        assert x_a_shapes == {(64, N_AUDIO_BINS)}, (
            f"STRONGEST-64 FAILED: {split_name} x_a shapes inconsistent: {x_a_shapes}, "
            f"expected only (64, {N_AUDIO_BINS})"
        )
print(f"[STRONGEST-64] Verified: every train/val/test window's audio feature array "
      f"has shape (n_chunks=64, {N_AUDIO_BINS}).")

print("\n" + "=" * 60)
print("     100% REAL PREPROCESSING COMPLETE (ZERO SYNTHETIC DATA)   ")
if _chunk_duration_log:
    _arr = np.array(_chunk_duration_log)
    print(f"Audio chunk duration check -> mean: {_arr.mean()*1000:.2f} ms | min: {_arr.min()*1000:.2f} ms | max: {_arr.max()*1000:.2f} ms")
print("=" * 60)
print(f"  • Total Train Diagnostic Windows : {len(train_samples)}")
print(f"  • Total Val Diagnostic Windows   : {len(val_samples)}")
print(f"  • Total Test Diagnostic Windows  : {len(test_samples)}")
print("=" * 60)

torch.save(train_samples, CACHE_DIR / "bimodal_train.pt")
torch.save(val_samples, CACHE_DIR / "bimodal_val.pt")
torch.save(test_samples, CACHE_DIR / "bimodal_test.pt")
print(f"✅ Successfully cached real bimodal dataset in {CACHE_DIR}")


# Cell 4b: Hann-Windowing Fairness Diagnostic (one-time, read-only — no training, pipeline unchanged)

In [ ]:
# ==============================================================================
# ONE-TIME DIAGNOSTIC (read-only, no training) — Hann-windowing fairness check.
#
# v4's Stage A ran "hann" with the strongest-64 bins that had been selected from
# RAW (unwindowed) spectra. This recomputes the selection from TRAINING-set
# audio twice — raw and Hann-windowed — with the exact same criterion as
# select_strongest_audio_bins (highest mean |rFFT| per bin, top 64, sorted), and
# prints both side by side. It does NOT change the pipeline: every split above
# was built with SELECTED_BIN_INDICES (raw), which stays frozen.
# ==============================================================================
def strongest_bins_raw_vs_hann(train_mission_dirs, n_bins=N_AUDIO_BINS):
    sums = {"raw": None, "hann": None}
    bin_count = 0
    fft_len_ref = None

    for m_dir in tqdm(train_mission_dirs, desc="Hann diagnostic (train-only audio scan)", leave=False):
        mission_data = parse_real_mission(m_dir)
        if mission_data is None:
            continue
        audio = mission_data["audio_chunks"]
        hann = np.hanning(audio.shape[1]).astype(audio.dtype)

        for key, frames in (("raw", audio), ("hann", audio * hann)):
            mag = np.abs(np.fft.rfft(frames, axis=1))
            min_len = mag.shape[1] if fft_len_ref is None else min(fft_len_ref, mag.shape[1])
            chunk_sum = mag[:, :min_len].sum(axis=0)
            sums[key] = chunk_sum if sums[key] is None else sums[key][:min_len] + chunk_sum
        fft_len_ref = min_len
        bin_count += audio.shape[0]

    return {key: np.sort(np.argsort(s / bin_count)[::-1][:n_bins]).astype(int) for key, s in sums.items()}

print("[HANN DIAGNOSTIC] Recomputing strongest-64 bin selection from TRAINING-SET audio only "
      f"({len(all_train_missions)} training missions; no validation/test audio used), twice: raw and Hann-windowed.")
_diag_bins = strongest_bins_raw_vs_hann(all_train_missions)
raw_bins, hann_bins = _diag_bins["raw"], _diag_bins["hann"]

raw_matches_pipeline = np.array_equal(raw_bins, SELECTED_BIN_INDICES)
print(f"[HANN DIAGNOSTIC] raw recomputation identical to the pipeline's frozen SELECTED_BIN_INDICES: "
      f"{raw_matches_pipeline} {'✅' if raw_matches_pipeline else '⚠️ (diagnostic does not reproduce the pipeline)'}")

raw_set, hann_set = set(raw_bins.tolist()), set(hann_bins.tolist())
overlap = raw_set & hann_set
print("\n  #  | raw bin (in use) | Hann bin   (* = also in the other selection)")
print("  ---+------------------+-----------")
for k, (r, h) in enumerate(zip(raw_bins.tolist(), hann_bins.tolist())):
    print(f"  {k:2d} | {r:4d}{'*' if r in hann_set else ' '}            | {h:4d}{'*' if h in raw_set else ' '}")

print(f"\n[HANN DIAGNOSTIC] OVERLAP: {len(overlap)}/{N_AUDIO_BINS} bins shared "
      f"({len(overlap) / N_AUDIO_BINS * 100:.1f}%)")
print(f"[HANN DIAGNOSTIC] only in RAW selection  ({len(raw_set - hann_set):2d}): {sorted(raw_set - hann_set)}")
print(f"[HANN DIAGNOSTIC] only in HANN selection ({len(hann_set - raw_set):2d}): {sorted(hann_set - raw_set)}")
if len(overlap) == N_AUDIO_BINS:
    print("[HANN DIAGNOSTIC] Identical selections -> v4's Hann run used the bins Hann itself would pick; "
          "that comparison was fair with respect to bin selection.")
else:
    print(f"[HANN DIAGNOSTIC] {N_AUDIO_BINS - len(overlap)} bin(s) differ -> v4's Hann run was fed bins chosen "
          "under the UNWINDOWED spectrum; a fair Hann re-test would re-select bins from Hann-windowed "
          "training audio. (Pipeline unchanged — informational only.)")

# Cell 5: DataLoader (weights_only=False for PyTorch 2.6+)

In [ ]:
class RealBiModalDataset(Dataset):
    def __init__(self, pt_path: Path, mean=None, std=None):
        self.samples = torch.load(pt_path, weights_only=False)
        self.Ta = 64
        self.Ts = 128

        if mean is None or std is None:
            all_s = np.concatenate([s["x_s"] for s in self.samples], axis=0)
            self.mean = all_s.mean(axis=0, keepdims=True)
            self.std = all_s.std(axis=0, keepdims=True) + 1e-6
        else:
            self.mean = mean
            self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        norm_sensor = (s["x_s"] - self.mean) / self.std

        # --- FIX 1: use the REAL per-window timestamps saved during
        # preprocessing (Cell 4), normalized by that window's real duration,
        # instead of a synthetic torch.linspace(0.0, 1.0, ...) that was
        # identical for every sample regardless of the underlying data.
        window_sec = float(s["window_sec"])
        tau_a = torch.tensor(s["tau_a"], dtype=torch.float32) / window_sec
        tau_s = torch.tensor(s["tau_s"], dtype=torch.float32) / window_sec

        return {
            "x_a": torch.tensor(s["x_a"], dtype=torch.float32),
            "x_s": torch.tensor(norm_sensor, dtype=torch.float32),
            "tau_a": tau_a,
            "tau_s": tau_s,
            "label": torch.tensor(s["label"], dtype=torch.long),
        }

train_ds = RealBiModalDataset(CACHE_DIR / "bimodal_train.pt")
val_ds   = RealBiModalDataset(CACHE_DIR / "bimodal_val.pt", mean=train_ds.mean, std=train_ds.std)
test_ds  = RealBiModalDataset(CACHE_DIR / "bimodal_test.pt", mean=train_ds.mean, std=train_ds.std)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=2)

print(f"DataLoaders ready -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)} windows")

In [ ]:
# ==============================================================================
# FIX 1 VERIFICATION HOOK — confirm tau_a / tau_s carry REAL, per-sample-varying
# timestamps, not the old synthetic torch.linspace(0.0, 1.0, ...) that was
# identical for every sample regardless of the underlying data.
# ==============================================================================
print("=" * 70)
print("VERIFICATION HOOK — FIX 1: real per-sample tau_a / tau_s")
print("=" * 70)

check_idxs = [0, 1, len(train_ds) // 2, len(train_ds) - 1]
taus_a, taus_s = [], []
for idx in check_idxs:
    item = train_ds[idx]
    taus_a.append(item["tau_a"])
    taus_s.append(item["tau_s"])
    print(f"sample idx={idx:>6} | tau_a[:5]={item['tau_a'][:5].numpy()} | tau_s[:5]={item['tau_s'][:5].numpy()}")

all_tau_a_identical = all(torch.allclose(taus_a[0], t) for t in taus_a[1:])
all_tau_s_identical = all(torch.allclose(taus_s[0], t) for t in taus_s[1:])

assert not all_tau_a_identical, (
    "❌ FIX 1 FAILED: tau_a is IDENTICAL across every sample checked — "
    "real timestamps are NOT reaching the model."
)
assert not all_tau_s_identical, (
    "❌ FIX 1 FAILED: tau_s is IDENTICAL across every sample checked — "
    "real timestamps are NOT reaching the model."
)
print("✅ FIX 1 VERIFIED: tau_a and tau_s vary across real samples (not synthetic/fixed).")
print("=" * 70)

# Cell 6: Training (class-weighted loss, seeded)

In [ ]:
set_all_seeds(SEED)  # identical weight init + batch order on every re-run

model = BiModalAeroSyncMamba(
    d_model=256,
    n_classes=5,
    num_mamba_blocks=4,
    dropout=0.15,
    temporal_bias_scale=1.0,
    audio_in_dim=64,
    sensor_in_dim=6,
).to(device)

# ==============================================================================
# FIX 2 VERIFICATION HOOK — print the EXACT class name of the fusion block in
# use, so it is unmissable whether training ran on the real mamba_ssm SSM or
# the pure-PyTorch fallback.
# ==============================================================================
_fusion_cls = type(model.mamba_blocks[0])
print("\n" + "=" * 70)
print(f"FUSION BACKBONE IN USE: {_fusion_cls.__module__}.{_fusion_cls.__name__}")
print(f"Resolution reason: {model.fusion_backbone_reason}")
print("=" * 70)
if _fusion_cls.__name__ == "VectorizedSSMBlock":
    print("🚨 WARNING: training with the pure-PyTorch FALLBACK block — this is NOT a real "
          "selective-scan SSM/Mamba (no hidden state, no A/B/C, no discretization step). 🚨")
else:
    print("✅ Training with the REAL mamba_ssm selective-scan SSM.")
print("=" * 70 + "\n")

# ==============================================================================
# FIX 4 VERIFICATION HOOK — inverse-frequency class weights computed from REAL
# training-set class counts (not hardcoded), printed per class.
# ==============================================================================
train_labels_arr = np.array([s["label"] for s in train_ds.samples])
class_counts = np.bincount(train_labels_arr, minlength=5)
class_weights_np = class_counts.sum() / (len(class_counts) * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

print("=" * 70)
print("FIX 4 VERIFICATION — inverse-frequency class weights (from real train counts)")
print("=" * 70)
for c in range(5):
    print(f"  Class {c}: count={class_counts[c]:>6} | weight={class_weights_np[c]:.4f}")
print("=" * 70 + "\n")

criterion = BiModalAeroSyncLoss(lambda_align=0.1, lambda_cons=0.1, class_weights=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

best_val_f1 = 0.0
best_weights = None
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_acc": []}

start_time = time.time()
print(f"🚀 Training Bi-Modal AeroSync-Mamba on {len(train_ds)} real flight windows...\n")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, running_cls = 0.0, 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs}", leave=False)
    for batch in pbar:
        xa = batch["x_a"].to(device, non_blocking=True)
        xs = batch["x_s"].to(device, non_blocking=True)
        ta = batch["tau_a"].to(device, non_blocking=True)
        ts = batch["tau_s"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
        loss, l_dict = criterion(logits, labels, pooled_reps, aux_preds)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        running_cls += l_dict["loss_cls"]
        pbar.set_postfix({"loss": f"{loss.item():.3f}", "cls": f"{l_dict['loss_cls']:.3f}"})

    scheduler.step()
    avg_train_loss = running_loss / len(train_loader)

    # --- Validation ---
    model.eval()
    val_preds, val_targets, val_probs = [], [], []
    val_loss_total = 0.0

    with torch.no_grad():
        for batch in val_loader:
            xa = batch["x_a"].to(device)
            xs = batch["x_s"].to(device)
            ta = batch["tau_a"].to(device)
            ts = batch["tau_s"].to(device)
            labels = batch["label"].to(device)

            logits, pooled_reps, aux_preds, _ = model(xa, xs, ta, ts)
            v_loss, _ = criterion(logits, labels, pooled_reps, aux_preds)
            val_loss_total += v_loss.item()

            probs = F.softmax(logits, dim=-1)
            val_preds.extend(probs.argmax(dim=-1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
            val_probs.extend(probs.cpu().numpy())

    avg_val_loss = val_loss_total / len(val_loader)
    val_metrics = compute_all_metrics(np.array(val_targets), np.array(val_preds), np.array(val_probs))

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_f1"].append(val_metrics["macro_f1"])
    history["val_acc"].append(val_metrics["accuracy"])

    is_best = ""
    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, CACHE_DIR / "aerosync_bimodal_best.pt")
        is_best = " ⭐ [Best Saved]"

    print(
        f"Epoch {epoch:02d}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f}{is_best}"
    )

elapsed = time.time() - start_time
print(f"\n🎉 Training Finished in {elapsed:.1f}s | Best Val Macro-F1: {best_val_f1:.4f}")

# Cell 7: Test Benchmark, Audio-Sensor Lag Analysis & Temporal-Shift Robustness

In [ ]:
if best_weights is not None:
    model.load_state_dict(best_weights)
model.eval()

# 1. Unseen Test Set Evaluation
test_preds, test_targets, test_probs, lags_as_list = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        xa = batch["x_a"].to(device)
        xs = batch["x_s"].to(device)
        ta = batch["tau_a"].to(device)
        ts = batch["tau_s"].to(device)
        labels = batch["label"].to(device)

        logits, _, _, aux = model(xa, xs, ta, ts)
        probs = F.softmax(logits, dim=-1)

        test_preds.extend(probs.argmax(dim=-1).cpu().numpy())
        test_targets.extend(labels.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())
        lags_as_list.extend(aux["lag_as"].cpu().numpy())

y_true = np.array(test_targets)
y_pred = np.array(test_preds)
y_prob = np.array(test_probs)

test_metrics = compute_all_metrics(y_true, y_pred, y_prob)

print("=" * 65)
print("     BI-MODAL AEROSYNC-MAMBA: 100% REAL TEST BENCHMARK        ")
print("=" * 65)
print(f"  • Test Accuracy       : {test_metrics['accuracy']:.4f} ({test_metrics['accuracy']*100:.2f}%)")
print(f"  • Macro-F1 Score      : {test_metrics['macro_f1']:.4f}")
print(f"  • Balanced Accuracy   : {test_metrics['balanced_accuracy']:.4f}")
print(f"  • One-vs-Rest AUROC   : {test_metrics['macro_auroc']:.4f}")
print(f"  • Learned Audio-IMU Lag (Δ_as) : {np.mean(lags_as_list):+.4f} (normalized units)")
print("=" * 65)

# 2. Hypothesis H2 Robustness (Artificial Temporal Shifts)
print("\n--- Hypothesis H2: Real Flight Shift Robustness (Audio <-> IMU Delay) ---")
delta_offsets = [0.0, 0.05, 0.10, 0.20, 0.30]
base_f1 = test_metrics["macro_f1"]
print(f"{'Offset (δ)':<16} | {'Accuracy':<10} | {'Macro-F1':<10} | {'AUROC':<10} | {'Retention':<10}")
print("-" * 62)

shift_results = []
for delta in delta_offsets:
    s_preds, s_targets, s_probs = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            xa = batch["x_a"].to(device)
            xs = batch["x_s"].to(device)
            ta = batch["tau_a"].to(device)
            ts_shifted = torch.clamp(batch["tau_s"].to(device) + delta, 0.0, 1.0)
            labels = batch["label"].to(device)

            logits, _, _, _ = model(xa, xs, ta, ts_shifted)
            probs = F.softmax(logits, dim=-1)
            s_preds.extend(probs.argmax(dim=-1).cpu().numpy())
            s_targets.extend(labels.cpu().numpy())
            s_probs.extend(probs.cpu().numpy())

    s_m = compute_all_metrics(np.array(s_targets), np.array(s_preds), np.array(s_probs))
    retention = (s_m["macro_f1"] / base_f1) * 100 if base_f1 > 0 else 100.0
    shift_results.append({
        "Offset_delta": "0.0 (Baseline)" if delta == 0.0 else f"+{delta:.2f}",
        "Accuracy": s_m["accuracy"],
        "Macro_F1": s_m["macro_f1"],
        "AUROC": s_m["macro_auroc"],
        "Retention_Percent": round(retention, 2),
    })

    label_str = f"+{delta:.2f}" if delta > 0 else "0.0 (Baseline)"
    print(f"{label_str:<16} | {s_m['accuracy']:<10.4f} | {s_m['macro_f1']:<10.4f} | {s_m['macro_auroc']:<10.4f} | {retention:>6.2f}%")

print("=" * 65)

# 3. Confusion Matrix Plot
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5), dpi=150)
plt.imshow(cm, cmap="Blues", interpolation="nearest")
plt.title("Bi-Modal Test Confusion Matrix (Real Audio + IMU)", fontsize=11, fontweight="bold")
plt.xlabel("Predicted Class (Broken Propellers)")
plt.ylabel("True Class (Broken Propellers)")
plt.colorbar()
for i in range(5):
    for j in range(5):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black" if cm[i, j] < cm.max()/2 else "white")
plt.tight_layout()
plt.show()

# Cell 7b: Class-Weighting Effect Check — per-class confusion matrix & recall (class-weighted model)

In [ ]:
# ==============================================================================
# CLASS-WEIGHTING EFFECT CHECK (part 1) — full confusion matrix + per-class
# precision / recall / F1 of the class-weighted model on the unseen test set.
# ==============================================================================
from sklearn.metrics import precision_recall_fscore_support

CLASS_NAMES = ["0 (Normal)", "1 Broken", "2 Broken", "3 Broken", "4 Broken"]

def print_per_class_report(y_true_, y_pred_, title):
    cm_ = confusion_matrix(y_true_, y_pred_, labels=list(range(5)))
    prec_, rec_, f1_, sup_ = precision_recall_fscore_support(
        y_true_, y_pred_, labels=list(range(5)), zero_division=0)

    print("=" * 72)
    print(f"  {title}")
    print("=" * 72)
    print("  Confusion matrix (rows = TRUE class, cols = PREDICTED class):")
    print("  " + " " * 12 + "".join(f"{'pred ' + str(c):>9}" for c in range(5)))
    for i in range(5):
        print(f"  {CLASS_NAMES[i]:<12}" + "".join(f"{cm_[i, j]:>9d}" for j in range(5)))
    print("\n  Per-class metrics:")
    print(f"  {'class':<12} {'precision':>10} {'recall':>10} {'f1':>10} {'support':>9}")
    for c in range(5):
        flag = "   <-- Class 4 (4 broken propellers)" if c == 4 else ""
        print(f"  {CLASS_NAMES[c]:<12} {prec_[c]:>10.4f} {rec_[c]:>10.4f} {f1_[c]:>10.4f} {sup_[c]:>9d}{flag}")
    print("=" * 72)
    return rec_, cm_

weighted_per_class_recall, weighted_cm = print_per_class_report(
    y_true, y_pred, "CLASS-WEIGHTED model (Cell 6, best-val checkpoint) — unseen test set")
weighted_test_metrics = dict(test_metrics)
print(f"Class 4 recall (class-weighted): {weighted_per_class_recall[4]:.4f}")

# Cell 8: Publication Figures (live Fig-5 retention legend)

In [ ]:
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support
from sklearn.manifold import TSNE

FIG_DIR = Path("/kaggle/working/bimodal_experiment_export/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

class_names = ["0 (Normal)", "1 Broken", "2 Broken", "3 Broken", "4 Broken"]
palette = ["#2b5c8f", "#d95f02", "#7570b3", "#e7298a", "#1b9e77"]

# ------------------------------------------------------------------------------
# Extract Latent Embeddings & Attention Weights from Test Set
# ------------------------------------------------------------------------------
all_zf, all_labels, all_preds, all_probs = [], [], [], []
all_lags, sample_attn_matrix = [], None

model.eval()
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        xa = batch["x_a"].to(device)
        xs = batch["x_s"].to(device)
        ta = batch["tau_a"].to(device)
        ts = batch["tau_s"].to(device)
        labels = batch["label"].to(device)

        # Encoders + HTT + LACA
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        Ha = model.htt(za, ta, 0)
        Hs = model.htt(zs, ts, 1)
        H_tilde_a, H_tilde_s, aux = model.laca(Ha, Hs, ta, ts)

        # Mamba fusion representation z_f
        sep = model.sep_token.expand(xa.shape[0], -1, -1)
        H_fused = torch.cat([H_tilde_a, sep, H_tilde_s], dim=1)
        F_seq = H_fused
        for blk in model.mamba_blocks: F_seq = blk(F_seq)
        weights = F.softmax(model.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = model.classifier(zf)
        probs = F.softmax(logits, dim=-1)

        all_zf.append(zf.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(probs.argmax(dim=-1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_lags.extend(aux["lag_as"].cpu().numpy())

        if sample_attn_matrix is None and len(xa) > 0:
            # Capture attention weights from first batch
            _, A_mn, _ = model.laca.laca_as(Ha, Hs, ta, ts)
            sample_attn_matrix = A_mn[0].cpu().numpy()

all_zf = np.concatenate(all_zf, axis=0)
all_probs = np.concatenate(all_probs, axis=0)
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_lags = np.array(all_lags)

# ==============================================================================
# Figure 1: Multi-Class One-vs-Rest ROC Curves
# ==============================================================================
plt.figure(figsize=(7, 6), dpi=300)
for i in range(5):
    y_binary = (all_labels == i).astype(int)
    fpr, tpr, _ = roc_curve(y_binary, all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=palette[i], lw=2.2, label=f"Class {class_names[i]} (AUC = {roc_auc:.4f})")

plt.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.6, label="Chance Level")
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.02])
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Sensitivity)", fontsize=11)
plt.title("Multi-Class ROC Curves (Real Audio + Telemetry)", fontsize=12, fontweight="bold", pad=12)
plt.legend(loc="lower right", fontsize=9, frameon=True)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_multiclass_roc_curves.png", dpi=300)
plt.savefig(FIG_DIR / "fig1_multiclass_roc_curves.pdf")
plt.show()

# ==============================================================================
# Figure 2: Per-Class Precision, Recall & F1 Breakdown
# ==============================================================================
prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average=None)
x = np.arange(5)
width = 0.25

plt.figure(figsize=(8, 5), dpi=300)
plt.bar(x - width, prec, width, label="Precision", color="#4c72b0", edgecolor="black", linewidth=0.5)
plt.bar(x, rec, width, label="Recall", color="#55a868", edgecolor="black", linewidth=0.5)
plt.bar(x + width, f1, width, label="F1-Score", color="#c44e52", edgecolor="black", linewidth=0.5)

plt.xlabel("Fault Severity (Broken Propeller Count)", fontsize=11)
plt.ylabel("Score", fontsize=11)
plt.title("Per-Class Diagnostic Performance Breakdown", fontsize=12, fontweight="bold", pad=12)
plt.xticks(x, class_names, fontsize=10)
plt.ylim([0.0, 1.0])
plt.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_per_class_metrics.png", dpi=300)
plt.savefig(FIG_DIR / "fig2_per_class_metrics.pdf")
plt.show()

# ==============================================================================
# Figure 3: LACA Cross-Attention Heatmap (Audio <-> Sensor Temporal Alignment)
# ==============================================================================
plt.figure(figsize=(7, 6), dpi=300)
im = plt.imshow(sample_attn_matrix, aspect="auto", cmap="viridis", origin="lower")
plt.title("LACA Cross-Attention Matrix (Audio Query vs IMU Key)", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("Normalized Sensor Telemetry Timesteps (Ts = 128)", fontsize=10)
plt.ylabel("Normalized Audio Timesteps (Ta = 64)", fontsize=10)
plt.colorbar(im, fraction=0.046, pad=0.04, label="Attention Weight")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_laca_attention_heatmap.png", dpi=300)
plt.savefig(FIG_DIR / "fig3_laca_attention_heatmap.pdf")
plt.show()

# ==============================================================================
# Figure 4: Per-Class Learned Lag Distribution (Δ_as)
# ==============================================================================
plt.figure(figsize=(7, 5), dpi=300)
lag_data_by_class = [all_lags[all_labels == c] for c in range(5)]
bp = plt.boxplot(lag_data_by_class, labels=class_names, patch_artist=True,
                 boxprops=dict(facecolor="#e0ecf4", color="#8856a7"),
                 medianprops=dict(color="#810f7c", lw=2))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

plt.axhline(0.0, color="gray", linestyle="--", alpha=0.7)
plt.title("Learned Inter-Modality Lag Distribution by Fault Class (Δ_as)", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("Fault Category", fontsize=11)
plt.ylabel("Learned Temporal Offset Δ_as (Normalized)", fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_lag_distribution_per_class.png", dpi=300)
plt.savefig(FIG_DIR / "fig4_lag_distribution_per_class.pdf")
plt.show()

# ==============================================================================
# Figure 5: Temporal Shift Robustness Curve (Validating Hypothesis H2)
# ==============================================================================
plt.figure(figsize=(7, 4.8), dpi=300)
shifts = [0.0, 0.05, 0.10, 0.20, 0.30]
shift_macro_f1 = [r["Macro_F1"] for r in shift_results]
shift_auroc    = [r["AUROC"] for r in shift_results]

# --- FIX 5: retention percentages computed LIVE from shift_results in this
# cell (previously hardcoded literal strings "Retention ≥ 99.6%" / "≥ 99.9%"
# that would go stale on any re-run). ---
f1_retention_min = (min(shift_macro_f1) / shift_macro_f1[0]) * 100
auroc_retention_min = (min(shift_auroc) / shift_auroc[0]) * 100
print(f"[FIX 5 VERIFICATION] Live-computed retention from shift_results in this cell -> "
      f"Macro-F1 >= {f1_retention_min:.2f}% | AUROC >= {auroc_retention_min:.2f}%")

plt.plot(shifts, shift_macro_f1, marker="o", lw=2.5, color="#1f77b4", markersize=7,
         label=f"Macro-F1 (Retention ≥ {f1_retention_min:.1f}%)")
plt.plot(shifts, shift_auroc, marker="s", lw=2.5, color="#2ca02c", markersize=7,
         label=f"AUROC (Retention ≥ {auroc_retention_min:.1f}%)")
plt.axhline(test_metrics["macro_f1"], color="gray", linestyle=":", alpha=0.6, label="Zero-Shift F1 Baseline")

plt.title("Hypothesis H2: Robustness Under Artificial Temporal Shifts (δ)", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("Injected Audio-Sensor Time Delay Offset (δ)", fontsize=11)
plt.ylabel("Evaluation Metric Score", fontsize=11)
plt.ylim([0.75, 1.00])
plt.legend(loc="upper right", fontsize=9.5, frameon=True)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_shift_robustness_curve.png", dpi=300)
plt.savefig(FIG_DIR / "fig5_shift_robustness_curve.pdf")
plt.show()

# ==============================================================================
# Figure 6: 2D t-SNE Latent Representation Space of Fused Embeddings (z_f)
# ==============================================================================
print("Computing 2D t-SNE projection of 1,131 test flight embeddings...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
zf_2d = tsne.fit_transform(all_zf)

plt.figure(figsize=(8, 6.5), dpi=300)
for c in range(5):
    idx = (all_labels == c)
    plt.scatter(zf_2d[idx, 0], zf_2d[idx, 1], c=palette[c], label=class_names[c], alpha=0.75, s=35, edgecolors="none")

plt.title("t-SNE Latent Space of AeroSync-Mamba Fused Embeddings (z_f)", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("t-SNE Dimension 1", fontsize=11)
plt.ylabel("t-SNE Dimension 2", fontsize=11)
plt.legend(title="Fault Severity", loc="best", fontsize=9.5)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig6_tsne_latent_clustering.png", dpi=300)
plt.savefig(FIG_DIR / "fig6_tsne_latent_clustering.pdf")
plt.show()

print("\n🎉 All 6 publication-quality figures generated and saved to:", FIG_DIR)

# Cell 9: Export Bi-Modal Experiment Artifacts to ZIP

In [ ]:
import os, json, zipfile, shutil
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

EXPORT_DIR = Path("/kaggle/working/bimodal_experiment_export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = EXPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save High-Res Confusion Matrix (300 DPI PNG & PDF for LaTeX)
plt.figure(figsize=(6, 5), dpi=300)
plt.imshow(cm, cmap="Blues", interpolation="nearest")
plt.title("Bi-Modal AeroSync-Mamba: Test Confusion Matrix", fontsize=11, fontweight="bold", pad=12)
plt.xlabel("Predicted Class (Broken Propellers)", fontsize=10)
plt.ylabel("True Class (Broken Propellers)", fontsize=10)
plt.colorbar(fraction=0.046, pad=0.04)
for i in range(5):
    for j in range(5):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", 
                 color="black" if cm[i, j] < cm.max()/2 else "white", fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "bimodal_confusion_matrix.png", dpi=300)
plt.savefig(FIG_DIR / "bimodal_confusion_matrix.pdf")
plt.close()

# 2. Save Metrics & Shift Table
with open(EXPORT_DIR / "real_bimodal_metrics.json", "w") as f:
    json.dump({
        "accuracy": float(test_metrics["accuracy"]),
        "macro_f1": float(test_metrics["macro_f1"]),
        "balanced_accuracy": float(test_metrics["balanced_accuracy"]),
        "macro_auroc": float(test_metrics["macro_auroc"]),
        "learned_lag_as": float(np.mean(lags_as_list)),
    }, f, indent=4)

shift_df = pd.DataFrame(shift_results)
shift_df.to_csv(EXPORT_DIR / "real_shift_robustness.csv", index=False)

# 3. Copy Best Model Checkpoint
best_ckpt = CACHE_DIR / "aerosync_bimodal_best.pt"
if best_ckpt.exists():
    shutil.copy(best_ckpt, EXPORT_DIR / "aerosync_bimodal_best.pt")

# 4. Create Downloadable ZIP
zip_path = Path("/kaggle/working/aerosync_bimodal_real_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(EXPORT_DIR):
        for file in files:
            p = Path(root) / file
            zipf.write(p, arcname=str(p.relative_to(EXPORT_DIR)))

print(f"✅ Download ready at: {zip_path}")
print(f"📦 Size: {zip_path.stat().st_size / (1024*1024):.2f} MB")

# Cell 10: Class-Weighting Effect Check — one unweighted comparison run (same seed / data / schedule)

In [ ]:
# ==============================================================================
# CLASS-WEIGHTING EFFECT CHECK (part 2) — ONE unweighted comparison run.
#
# Identical to Cell 6 in every respect (same SEED reset right before model
# construction, same model / optimizer / scheduler / epochs, same train / val /
# test loaders and therefore same batch order, same best-val-macro-F1
# checkpointing, same test evaluation) EXCEPT the loss: BiModalAeroSyncLoss
# with class_weights=None. Runs after the export cell and uses only local names,
# so the class-weighted model, figures and ZIP above are untouched.
# ==============================================================================
import gc

def train_and_test_unweighted():
    set_all_seeds(SEED)
    model_u = BiModalAeroSyncMamba(
        d_model=256,
        n_classes=5,
        num_mamba_blocks=4,
        dropout=0.15,
        temporal_bias_scale=1.0,
        audio_in_dim=64,
        sensor_in_dim=6,
    ).to(device)
    criterion_u = BiModalAeroSyncLoss(lambda_align=0.1, lambda_cons=0.1)   # <-- NO class weights
    optimizer_u = torch.optim.AdamW(model_u.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler_u = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_u, T_max=epochs, eta_min=1e-6)

    best_f1_u, best_w_u = 0.0, None
    t0 = time.time()
    print(f"🚀 UNWEIGHTED comparison run (seed={SEED}, {epochs} epochs, same loaders as Cell 6)...\n")
    try:
        for epoch in range(1, epochs + 1):
            model_u.train()
            running_loss = 0.0
            for batch in tqdm(train_loader, desc=f"[unweighted] Epoch {epoch:02d}/{epochs}", leave=False):
                xa = batch["x_a"].to(device, non_blocking=True)
                xs = batch["x_s"].to(device, non_blocking=True)
                ta = batch["tau_a"].to(device, non_blocking=True)
                ts = batch["tau_s"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                optimizer_u.zero_grad()
                logits, pooled_reps, aux_preds, _ = model_u(xa, xs, ta, ts)
                loss, _ = criterion_u(logits, labels, pooled_reps, aux_preds)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_u.parameters(), max_norm=1.0)
                optimizer_u.step()
                running_loss += loss.item()
            scheduler_u.step()

            model_u.eval()
            v_preds, v_targets, v_probs = [], [], []
            with torch.no_grad():
                for batch in val_loader:
                    logits, _, _, _ = model_u(batch["x_a"].to(device), batch["x_s"].to(device),
                                              batch["tau_a"].to(device), batch["tau_s"].to(device))
                    probs = F.softmax(logits, dim=-1)
                    v_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                    v_targets.extend(batch["label"].numpy())
                    v_probs.extend(probs.cpu().numpy())
            vm = compute_all_metrics(np.array(v_targets), np.array(v_preds), np.array(v_probs))
            is_best = ""
            if vm["macro_f1"] > best_f1_u:
                best_f1_u = vm["macro_f1"]
                best_w_u = copy.deepcopy(model_u.state_dict())
                is_best = " ⭐ [Best]"
            print(f"[unweighted] Epoch {epoch:02d}/{epochs} | Train Loss: {running_loss / len(train_loader):.4f} | "
                  f"Val Acc: {vm['accuracy']:.4f} | Val Macro-F1: {vm['macro_f1']:.4f}{is_best}")
        print(f"\n[unweighted] Training finished in {time.time() - t0:.1f}s | Best Val Macro-F1: {best_f1_u:.4f}")

        if best_w_u is not None:
            model_u.load_state_dict(best_w_u)
        model_u.eval()
        t_preds, t_targets, t_probs = [], [], []
        with torch.no_grad():
            for batch in test_loader:
                logits, _, _, _ = model_u(batch["x_a"].to(device), batch["x_s"].to(device),
                                          batch["tau_a"].to(device), batch["tau_s"].to(device))
                probs = F.softmax(logits, dim=-1)
                t_preds.extend(probs.argmax(dim=-1).cpu().numpy())
                t_targets.extend(batch["label"].numpy())
                t_probs.extend(probs.cpu().numpy())
        return np.array(t_targets), np.array(t_preds), np.array(t_probs)
    finally:
        del model_u, optimizer_u, scheduler_u, criterion_u, best_w_u
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

y_true_u, y_pred_u, y_prob_u = train_and_test_unweighted()
unweighted_test_metrics = compute_all_metrics(y_true_u, y_pred_u, y_prob_u)
unweighted_per_class_recall, _ = print_per_class_report(
    y_true_u, y_pred_u, "UNWEIGHTED comparison model (same seed / data / schedule) — unseen test set")

print("\n" + "=" * 72)
print("  CLASS-WEIGHTING BEFORE / AFTER — same seed, same data, only the loss differs")
print("=" * 72)
print(f"  {'':<22} {'unweighted':>12} {'class-weighted':>15} {'Δ (weighted - unw.)':>21}")
for c in range(5):
    u, w = unweighted_per_class_recall[c], weighted_per_class_recall[c]
    flag = "  <-- Class 4" if c == 4 else ""
    print(f"  recall {CLASS_NAMES[c]:<15} {u:>12.4f} {w:>15.4f} {w - u:>+21.4f}{flag}")
for key in ("accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"):
    u, w = unweighted_test_metrics[key], weighted_test_metrics[key]
    print(f"  {key:<22} {u:>12.4f} {w:>15.4f} {w - u:>+21.4f}")
print("=" * 72)
d4 = weighted_per_class_recall[4] - unweighted_per_class_recall[4]
print(f"Class 4 recall: unweighted {unweighted_per_class_recall[4]:.4f} -> class-weighted "
      f"{weighted_per_class_recall[4]:.4f} ({d4:+.4f}) => class weighting "
      f"{'IMPROVES' if d4 > 0 else ('does NOT change' if d4 == 0 else 'LOWERS')} Class 4 recall in this run.")